# ⚠️ Important Warning: Rate Limiting

**YouTube may rate-limit or temporarily ban your IP when fetching large amounts of subtitle data.**

## Recommended Solutions:
- **Use a rotating proxy service** to distribute requests across multiple IPs
  - [WebShare](https://www.webshare.io/) - Professional proxy rotation service

**⚠️ Proceed with caution and always respect YouTube's Terms of Service.**

📚 **Additional Resources:**
- [YouTube Transcript API Documentation](https://github.com/jdepoix/youtube-transcript-api)

In [ ]:
from pathlib import Path

import polars as pl
from loguru import logger

from scrapetube.utils.config import (
    SUBTITLES_PARQUET,
    get_timestamp_string
)
from scrapetube.cc.subtitle import fetch_video_transcripts_concurrent
from scrapetube.utils.logger import setup_logging

pl.Config.set_fmt_str_lengths(100)

today_string = get_timestamp_string()
setup_logging(write_to_file=False)
data_dir = Path("../data")

In [ ]:
def load_latest_data(prefix:str, data_dir:Path):
    """Load the most recently modified parquet file with the given prefix from data directory."""
    latest_file = max(data_dir.glob(f"{prefix}*.parquet"), key=lambda f: f.stat().st_mtime)
    logger.info(f"Loaded {latest_file}")
    df = pl.read_parquet(latest_file)
    return df

In [ ]:
channel_video_meta_df = load_latest_data("channel_video_meta", data_dir)
channel_video_meta_df

In [ ]:
_ = fetch_video_transcripts_concurrent(
    video_ids=channel_video_meta_df["video_id"].to_list(),
    file_path_parquet=SUBTITLES_PARQUET,
    sleep=(5, 25),
)